In [1]:
!pip install -q llama-index

In [7]:
import sys
from pathlib import Path
import pickle
import time
import logging
import numpy as np
from scipy.sparse import csr_matrix, save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from llama_index.core import Document

# Add parent directory to Python path
# Assuming your project root is `/work`, add it to the system path
project_root = Path('/work')
sys.path.append(str(project_root))
from src.utils.data_utils import load_documents

# Function to load vocabulary from file
def load_vocabulary(vocab_path):
    """Load vocabulary dictionary from pickle file"""
    try:
        with open(vocab_path, 'rb') as f:
            vocab_dict = pickle.load(f)
        print(f"Loaded vocabulary with {len(vocab_dict)} terms from {vocab_path}")
        return vocab_dict
    except Exception as e:
        print(f"Error loading vocabulary from {vocab_path}: {e}")
        return None

# Custom TF-IDF vectorizer wrapper for consistency with BM25 interface
class SparseTFIDFVectorizer:
    def __init__(self, norm='l2', use_idf=True, smooth_idf=True, sublinear_tf=False, vocabulary=None):
        self.norm = norm
        self.use_idf = use_idf
        self.smooth_idf = smooth_idf
        self.sublinear_tf = sublinear_tf
        self.vocabulary = vocabulary
        
        # Include vocabulary in TfidfVectorizer if provided
        vectorizer_params = {
            'binary': False, 
            'lowercase': True, 
            'token_pattern': r'\b\w+\b',
            'norm': self.norm,
            'use_idf': self.use_idf,
            'smooth_idf': self.smooth_idf,
            'sublinear_tf': self.sublinear_tf
        }

        # Add vocabulary parameter if provided
        if self.vocabulary is not None:
            vectorizer_params['vocabulary'] = self.vocabulary
            
        self.vectorizer = TfidfVectorizer(**vectorizer_params)

    def fit(self, documents):
        texts = [doc.text if isinstance(doc, Document) else doc for doc in documents]
        self.vectorizer.fit(texts)
        return self

    def transform(self, documents):
        if isinstance(documents, str):
            documents = [documents]
        texts = [doc.text if isinstance(doc, Document) else doc for doc in documents]
        return self.vectorizer.transform(texts)

# Helper function to extract text from documents
def extract_texts(documents):
    return [doc.text if isinstance(doc, Document) else doc for doc in documents]

# Function to save index components.
def save_index(index_path: Path, vectorizer: SparseTFIDFVectorizer, embeddings, documents):
    index_path.mkdir(parents=True, exist_ok=True)
    
    # Save the underlying TfidfVectorizer instead of the wrapper
    with open(index_path / "vectorizer.pkl", "wb") as f:
        pickle.dump(vectorizer.vectorizer, f)
    
    save_npz(index_path / "embeddings.npz", embeddings)
    
    with open(index_path / "documents.pkl", "wb") as f:
        pickle.dump(documents, f)
    
    logging.info(f"Index saved to {index_path}")

# Main function with progress updates and estimated time remaining.
def main(file, restricted_vocab_path=None, output_suffix=""):
    start = time.time()
    print("Starting TF-IDF indexing process...")

    # Handle restricted vocabulary
    allowed_vocab_dict = None
    if restricted_vocab_path:
        allowed_vocab_dict = load_vocabulary(restricted_vocab_path)
        if allowed_vocab_dict is None:
            print("Failed to load vocabulary. Proceeding with unrestricted vocabulary.")
        else:
            print(f"Using restricted vocabulary with {len(allowed_vocab_dict)} terms")
    else:
        print("Using unrestricted vocabulary")
 
    print("Setting up file paths...")
    documents_path_texto = Path('/work/data/processed/' + file + '_texto.pkl')
    documents_path_resumen = Path('/work/data/processed/' + file + '_resumen.pkl')

    # Add suffix to index paths if using restricted vocabulary
    base_suffix = output_suffix if output_suffix else ("_restricted" if allowed_vocab_dict else "")
    index_path_texto = Path(f'/work/sandbox/tf_idf/indexes/texto{base_suffix}')
    index_path_resumen = Path(f'/work/sandbox/tf_idf/indexes/resumen{base_suffix}')

    print("Loading documents for 'texto'...")
    documents_texto = load_documents(documents_path_texto)
    print("Loading documents for 'resumen'...")
    documents_resumen = load_documents(documents_path_resumen)

    if documents_texto is None or documents_resumen is None:
        logging.error("Failed to load one or both document sets. Exiting.")
        return

    datasets = [
        ("texto", documents_texto, index_path_texto),
        ("resumen", documents_resumen, index_path_resumen)
    ]
    total_steps = len(datasets)

    for i, (name, docs, index_path) in enumerate(datasets, 1):
        print(f"\nProcessing {name} documents ({i}/{total_steps})...")
        step_start = time.time()
        
        print("Fitting TF-IDF vectorizer...")

        # Use vocabulary parameter if provided
        vectorizer_params = {
            'binary': False, 
            'lowercase': True, 
            'token_pattern': r'\b\w+\b',
            'norm': 'l2',
            'use_idf': True,
            'smooth_idf': True,
            'sublinear_tf': False
        }

        # Use SparseTFIDFVectorizer with optional vocabulary
        if allowed_vocab_dict:
            print(f"  Using restricted vocabulary with {len(allowed_vocab_dict)} terms")
            vectorizer = SparseTFIDFVectorizer(vocabulary=allowed_vocab_dict)
        else:
            print("  Using unrestricted vocabulary")
            vectorizer = SparseTFIDFVectorizer()
        
        # Use the wrapper's fit method
        vectorizer.fit(docs)
        
        print("Transforming documents to TF-IDF embeddings...")
        embeddings = vectorizer.transform(docs)

        print("Saving index for", name, "documents...")
        save_index(index_path, vectorizer, embeddings, docs)

        step_end = time.time()
        step_elapsed = step_end - step_start
        remaining_steps = total_steps - i
        estimated_remaining = step_elapsed * remaining_steps if remaining_steps > 0 else 0
        minutes, seconds = divmod(estimated_remaining, 60)
        print(f"Completed processing {name} documents in {step_elapsed:.2f} seconds. "
              f"Estimated time remaining: {int(minutes)} minutes, {int(seconds)} seconds.")

    end = time.time()
    total_time = end - start
    print(f"\nTotal execution time: {total_time:.2f} seconds")

In [8]:
main('OEB')

Starting TF-IDF indexing process...
Using unrestricted vocabulary
Setting up file paths...
Loading documents for 'texto'...
Loading documents for 'resumen'...

Processing texto documents (1/2)...
Fitting TF-IDF vectorizer...
  Using unrestricted vocabulary
Transforming documents to TF-IDF embeddings...
Saving index for texto documents...
Completed processing texto documents in 4.19 seconds. Estimated time remaining: 0 minutes, 4 seconds.

Processing resumen documents (2/2)...
Fitting TF-IDF vectorizer...
  Using unrestricted vocabulary
Transforming documents to TF-IDF embeddings...
Saving index for resumen documents...
Completed processing resumen documents in 2.23 seconds. Estimated time remaining: 0 minutes, 0 seconds.

Total execution time: 10.96 seconds


In [11]:
vocab_path = Path('/work/sandbox/tf_idf/vocab_dict.pkl')
main('OEB', restricted_vocab_path=vocab_path)

Starting TF-IDF indexing process...
Loaded vocabulary with 153 terms from /work/sandbox/tf_idf/vocab_dict.pkl
Using restricted vocabulary with 153 terms
Setting up file paths...
Loading documents for 'texto'...
Loading documents for 'resumen'...

Processing texto documents (1/2)...
Fitting TF-IDF vectorizer...
  Using restricted vocabulary with 153 terms
Transforming documents to TF-IDF embeddings...
Saving index for texto documents...
Completed processing texto documents in 3.76 seconds. Estimated time remaining: 0 minutes, 3 seconds.

Processing resumen documents (2/2)...
Fitting TF-IDF vectorizer...
  Using restricted vocabulary with 153 terms
Transforming documents to TF-IDF embeddings...
Saving index for resumen documents...
Completed processing resumen documents in 2.06 seconds. Estimated time remaining: 0 minutes, 0 seconds.

Total execution time: 9.93 seconds
